# SDS PySpark Tutorial — Part 3
## SDS Comprehensive-Exam Algorithm Patterns

This notebook connects the Spark mechanics from Parts 1–2 to the actual SDS algorithm families.

The goal is **not** to reteach every algorithm from Lessons 1–10.

The goal is:

> When you open an algorithm notebook during the exam, understand what the Spark code is doing and where the scalable computation lives.

---

## Official SDS areas covered

The exam coverage has four equal blocks:

1. hashing / locality-sensitive hashing;
2. stream mining;
3. link analysis;
4. social-network graph mining.

This notebook emphasizes the Spark implementation patterns most likely to recur across those blocks.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T

spark = SparkSession.builder.appName("SDS-PySpark-Part3").getOrCreate()
print("Spark version:", spark.version)

# 1. Exam pattern map: prompt → Spark shape

| Exam idea | Spark shape to recognize |
|---|---|
| stable deterministic sample | `xxhash64` → `pmod` → threshold |
| frequency/count by key | `groupBy(key).agg(...)` |
| LSH candidate buckets | compute bucket key → `groupBy`/self-join candidates |
| PageRank | join source rank to edges → group by destination |
| HITS authority | join hub(source) to edges → group by destination |
| HITS hub | join authority(destination) to edges → group by source |
| triangles/clustering | symmetrize → self-join 2-hop → join closing edge |
| modularity scoring | community labels + original edges/degrees → aggregate |
| iterative methods | cache reused state → scalar residual → repeat |
| SimRank full pair state | recognize \(O(n^2)\); bound/reduce if necessary |
| spectral method | sparse edge-based matrix-vector products, not dense \(n\times n\) |

If you understand the row flow behind these patterns, the code becomes much easier to reconstruct.

# 2. Stable deterministic hash sampling

Prompt cue:

> Keep the **same fraction of entities** every run without storing a selected-ID list.

The key is to hash the **entity identity**.

Conceptually:

```text
user_id
   ↓ stable hash
integer
   ↓ positive modulo
bucket 0 ... B-1
   ↓ threshold
keep / discard
```

In [ ]:
events = spark.createDataFrame([
    ("u1", 1),
    ("u1", 2),
    ("u2", 3),
    ("u2", 4),
    ("u3", 5),
    ("u4", 6),
    ("u5", 7),
], ["user_id", "event_id"])

B = 10_000
fraction = 0.40
T = int(B * fraction)

sampled = (
    events
    .withColumn(
        "bucket",
        F.pmod(
            F.xxhash64(F.concat(F.lit("SDS|"), F.col("user_id"))),
            F.lit(B)
        )
    )
    .withColumn("keep", F.col("bucket") < F.lit(T))
)

sampled.show()

## Why this is correct for stable entity sampling

Every event with the same `user_id` hashes to the same bucket.

So all rows for `u1` receive one consistent decision.

If you hashed `user_id + event_id`, you would be sampling **events**, not stable entities.

### Fast sanity check

```python
sampled.groupBy("user_id").agg(F.countDistinct("keep"))
```

Every user should have exactly one distinct keep decision.

In [ ]:
sampled.groupBy("user_id").agg(
    F.countDistinct("keep").alias("distinct_decisions")
).show()

# 3. LSH in Spark: understand candidate generation

The algorithm-specific notebook contains the math.

The Spark idea is:

```text
object
  ↓ compute multiple hash/group keys
(object_id, group_id, bucket_key)
  ↓
objects sharing same (group_id, bucket_key)
  ↓
candidate pairs
  ↓
exact similarity only on candidates
```

The scalable purpose of LSH is to avoid comparing **every object to every other object**.

## 3.1 A tiny candidate-bucket example

Here the bucket keys are already supplied so we can focus on Spark mechanics.

In [ ]:
bucket_rows = spark.createDataFrame([
    ("d1", 0, "A"),
    ("d2", 0, "A"),   # d1,d2 collide in group 0
    ("d3", 0, "B"),
    ("d1", 1, "X"),
    ("d2", 1, "Y"),
    ("d3", 1, "X"),   # d1,d3 collide in group 1
], ["doc_id", "group_id", "bucket_key"])

bucket_rows.show()

To generate candidate pairs, self-join rows that share the same group and bucket.

Use `a.doc_id < b.doc_id` to avoid self-pairs and duplicate reversed pairs.

In [ ]:
candidates = (
    bucket_rows.alias("a")
    .join(
        bucket_rows.alias("b"),
        (F.col("a.group_id") == F.col("b.group_id")) &
        (F.col("a.bucket_key") == F.col("b.bucket_key"))
    )
    .filter(F.col("a.doc_id") < F.col("b.doc_id"))
    .select(
        F.col("a.doc_id").alias("left"),
        F.col("b.doc_id").alias("right")
    )
    .distinct()
)

candidates.show()

### Scalability trap

If one bucket contains \(k\) objects, all candidate pairs can be:

\[
\frac{k(k-1)}{2}
\]

A badly skewed bucket can therefore explode.

LSH reduces all-pairs work only when the bucket structure is sufficiently selective.

# 4. Stream-mining implementation perspective

The stream-mining lessons include:

- sampling/filtering;
- distinct counting;
- frequency estimation / CMS;
- moments / AMS;
- DGIM;
- decaying windows.

The algorithm notebooks contain the sketch-specific state rules.

From a Spark perspective, remember two different patterns:

### Pattern A — distributed batch reduction

Example: exact frequency baseline.

```python
events.groupBy("key").count()
```

### Pattern B — bounded sketch state

A sketch intentionally keeps a **small summary** instead of all historical rows.

The important scalability principle is not “everything must be a giant DataFrame.”  
It is:

> **Do not keep the entire stream history when the algorithm is specifically designed to keep bounded state.**

This Spark tutorial does not replace the dedicated CMS/HLL/AMS/DGIM notebooks; it teaches the distributed-data mechanics needed around them.

In [ ]:
stream = spark.createDataFrame([
    ("A",), ("A",), ("B",), ("C",), ("A",), ("B",)
], ["key"])

# Exact frequency baseline — useful to verify a sketch on a toy dataset.
stream.groupBy("key").count().orderBy("key").show()

# 5. PageRank — first trace one iteration by hand

Graph:

```text
A → B
A → C
B → C
C → A
C → D
D → C
E → C
```

PageRank needs:

1. all nodes;
2. out-degree;
3. current rank;
4. one contribution per edge;
5. aggregation by destination;
6. dangling mass;
7. teleportation;
8. convergence residual.

Spark's role is to keep the edge contribution and aggregation distributed.

In [ ]:
edges = spark.createDataFrame([
    ("A", "B"),
    ("A", "C"),
    ("B", "C"),
    ("C", "A"),
    ("C", "D"),
    ("D", "C"),
    ("E", "C"),
], ["src", "dst"]).distinct()

nodes = (
    edges.select(F.col("src").alias("id"))
    .union(edges.select(F.col("dst").alias("id")))
    .distinct()
    .cache()
)

outdeg = edges.groupBy("src").agg(F.count("*").alias("outdeg")).cache()

N = nodes.count()
ranks = nodes.withColumn("rank", F.lit(1.0 / N)).cache()

print("N =", N)
ranks.orderBy("id").show()

## 5.1 One PageRank message row

If node A has rank \(r(A)\) and out-degree 2, each outgoing edge carries:

\[
r(A)/2
\]

So joining `edges` with `ranks` and `outdeg` creates one message row per edge.

In [ ]:
messages = (
    edges.alias("e")
    .join(ranks.alias("r"), F.col("e.src") == F.col("r.id"))
    .join(outdeg.alias("o"), F.col("e.src") == F.col("o.src"))
    .select(
        F.col("e.src").alias("src"),
        F.col("e.dst").alias("dst"),
        (F.col("r.rank") / F.col("o.outdeg")).alias("contrib")
    )
)

messages.orderBy("src", "dst").show()

## 5.2 Aggregate by destination

Several source nodes can contribute to the same destination.

That is why PageRank uses:

```text
groupBy(dst).sum(contrib)
```

In [ ]:
incoming = (
    messages.groupBy("dst")
            .agg(F.sum("contrib").alias("incoming"))
)

incoming.orderBy("dst").show()

## 5.3 Full PageRank step with dangling handling

Even if this toy graph has no dangling node, the reusable implementation should handle them.

In [ ]:
def pagerank_step(edges, nodes, outdeg, ranks, N, damping=0.85):
    dangling = (
        ranks.join(outdeg, ranks.id == outdeg.src, "left")
             .filter(F.col("outdeg").isNull())
             .agg(F.sum("rank").alias("D"))
             .first()["D"]
    ) or 0.0

    incoming = (
        edges.alias("e")
        .join(ranks.alias("r"), F.col("e.src") == F.col("r.id"))
        .join(outdeg.alias("o"), F.col("e.src") == F.col("o.src"))
        .select(
            F.col("e.dst").alias("id"),
            (F.col("r.rank") / F.col("o.outdeg")).alias("contrib")
        )
        .groupBy("id")
        .agg(F.sum("contrib").alias("incoming"))
    )

    next_ranks = (
        nodes.join(incoming, "id", "left")
             .fillna(0.0, subset=["incoming"])
             .withColumn(
                 "rank",
                 F.lit((1.0 - damping) / N)
                 + F.lit(damping) *
                   (F.col("incoming") + F.lit(dangling / N))
             )
             .select("id", "rank")
    )

    return next_ranks

In [ ]:
MAX_IT = 50
TOL = 1e-8

for it in range(MAX_IT):
    nxt = pagerank_step(edges, nodes, outdeg, ranks, N).cache()

    residual = (
        nxt.alias("n")
        .join(ranks.alias("o"), "id")
        .agg(F.sum(F.abs(F.col("n.rank") - F.col("o.rank"))).alias("l1"))
        .first()["l1"]
    )

    ranks.unpersist()
    ranks = nxt

    if residual < TOL:
        print("Converged at iteration", it + 1, "residual =", residual)
        break

ranks.orderBy(F.desc("rank")).show()

rank_sum = ranks.agg(F.sum("rank").alias("s")).first()["s"]
print("Rank sum =", rank_sum)

## PageRank exam checks

- all nodes preserved;
- rank sum \(\approx 1\);
- source rank divided by out-degree (or outgoing weight sum);
- dangling mass handled;
- residual reported;
- final top-k may be collected/displayed;
- full edge table remains distributed.

# 6. Topic-sensitive / personalized PageRank

The Spark structure is almost identical.

The main conceptual change is the teleport vector \(v\).

Instead of uniform teleportation:

```text
1/N for every node
```

you may assign higher/nonzero teleport probability to topic seed nodes.

Spark implementation pattern:

```text
nodes[id, v]
   JOIN incoming[id, incoming]
   ↓
rank = (1-d)*v + d*(incoming + dangling*v)
```

The important exam detail is that personalized dangling mass should follow the same \(v\), not automatically uniform \(1/N\).

In [ ]:
topic_v = spark.createDataFrame([
    ("A", 0.5),
    ("B", 0.5),
    ("C", 0.0),
    ("D", 0.0),
    ("E", 0.0),
], ["id", "v"])

topic_v.agg(F.sum("v").alias("sum_v")).show()

# 7. HITS — understand the two data flows

HITS maintains two scores.

Authority:

> a node is authoritative if strong hubs point **to it**.

Hub:

> a node is a strong hub if it points **to strong authorities**.

Spark makes the two recurrences look symmetric.

## 7.1 Authority update

```text
edges[src,dst]
     JOIN
hub[id,hub] on src=id
     ↓
one hub value per edge
     ↓
GROUP BY dst
     ↓
SUM hub
```

In [ ]:
hits_nodes = nodes
hubs = hits_nodes.withColumn("hub", F.lit(1.0))

auth_raw = (
    edges.alias("e")
    .join(hubs.alias("h"), F.col("e.src") == F.col("h.id"))
    .groupBy(F.col("e.dst").alias("id"))
    .agg(F.sum(F.col("h.hub")).alias("auth"))
)

auth = (
    hits_nodes.join(auth_raw, "id", "left")
              .fillna(0.0, subset=["auth"])
)

auth.orderBy("id").show()

## 7.2 Why join back to `hits_nodes`?

A node can legitimately have authority 0.

If `auth_raw` lacks that node and you keep only `auth_raw`, the node disappears from state.

The full-node left join preserves it.

## 7.3 Hub update

Now reverse the role:

```text
edges[src,dst]
     JOIN
authority[id,auth] on dst=id
     ↓
GROUP BY src
     ↓
SUM auth
```

In [ ]:
hub_raw = (
    edges.alias("e")
    .join(auth.alias("a"), F.col("e.dst") == F.col("a.id"))
    .groupBy(F.col("e.src").alias("id"))
    .agg(F.sum(F.col("a.auth")).alias("hub"))
)

hub = (
    hits_nodes.join(hub_raw, "id", "left")
              .fillna(0.0, subset=["hub"])
)

hub.orderBy("id").show()

## HITS exam checklist

- authority groups by **destination**;
- hub groups by **source**;
- preserve all nodes;
- L2-normalize each score vector;
- compute convergence residual;
- keep edges distributed;
- collect only norms/residuals and final top-k.

# 8. Triangle counting and global clustering

We now apply the self-join from Part 2.

For a simple undirected graph:

1. remove self-loops;
2. canonicalize edges;
3. deduplicate;
4. symmetrize;
5. create ordered two-hop paths;
6. join a closing edge.

Each undirected triangle appears **6 times** as an ordered closed two-hop path.

In [ ]:
raw = spark.createDataFrame([
    ("A", "B"),
    ("B", "C"),
    ("C", "A"),   # triangle ABC
    ("C", "D"),
], ["u", "v"])

canon = (
    raw.filter(F.col("u") != F.col("v"))
       .select(
           F.least("u", "v").alias("u"),
           F.greatest("u", "v").alias("v")
       )
       .distinct()
)

E = (
    canon.select(F.col("u").alias("src"), F.col("v").alias("dst"))
         .union(canon.select(F.col("v").alias("src"), F.col("u").alias("dst")))
         .distinct()
         .cache()
)

E.show()

In [ ]:
two_hop = (
    E.alias("e1")
    .join(E.alias("e2"), F.col("e1.dst") == F.col("e2.src"))
    .select(
        F.col("e1.src").alias("u"),
        F.col("e1.dst").alias("v"),
        F.col("e2.dst").alias("w")
    )
    .filter(F.col("u") != F.col("w"))
)

two_hop.orderBy("u", "v", "w").show()

A two-hop path \(u\to v\to w\) is **not automatically a triangle**.

We need the closing edge \(u\to w\).

In [ ]:
closed = (
    two_hop.alias("p")
    .join(
        E.alias("e3"),
        (F.col("p.u") == F.col("e3.src")) &
        (F.col("p.w") == F.col("e3.dst"))
    )
)

ordered_wedges = two_hop.count()
ordered_closed = closed.count()

triangles = ordered_closed / 6
global_clustering = ordered_closed / ordered_wedges if ordered_wedges else 0.0

print("ordered_wedges =", ordered_wedges)
print("ordered_closed =", ordered_closed)
print("triangles =", triangles)
print("global clustering =", global_clustering)

## Why the clustering formula works in this representation

If \(W\) is the number of unordered wedges and \(T\) triangles:

- ordered wedges = \(2W\);
- ordered closed wedges = \(6T\).

Therefore:

\[
\frac{6T}{2W} = \frac{3T}{W}.
\]

The exam trap is a **counting convention mismatch**, not just a coding error.

# 9. Connected components without GraphFrames

Because external graph packages may be disallowed, a simple built-in Spark pattern is minimum-label propagation.

Conceptually:

```text
every node label = its own ID
        ↓
send labels across edges
        ↓
each node keeps minimum label seen
        ↓
repeat until no label changes
```

This is useful for:

- connected components themselves;
- communities after thresholding a similarity graph;
- Girvan-Newman partitions after removing edges.

In [ ]:
cc_nodes = (
    E.select(F.col("src").alias("id"))
     .union(E.select(F.col("dst").alias("id")))
     .distinct()
)

labels = cc_nodes.withColumn("label", F.col("id"))

MAX_IT = 50

for it in range(MAX_IT):
    msgs = (
        E.alias("e")
         .join(labels.alias("l"), F.col("e.src") == F.col("l.id"))
         .select(
             F.col("e.dst").alias("id"),
             F.col("l.label").alias("candidate")
         )
         .groupBy("id")
         .agg(F.min("candidate").alias("nbr_min"))
    )

    new_labels = (
        labels.alias("old")
              .join(msgs.alias("m"), "id", "left")
              .select(
                  "id",
                  F.least(
                      F.col("old.label"),
                      F.coalesce(F.col("m.nbr_min"), F.col("old.label"))
                  ).alias("label")
              )
    )

    changed = (
        new_labels.alias("n")
                  .join(labels.alias("o"), "id")
                  .filter(F.col("n.label") != F.col("o.label"))
                  .limit(1)
                  .count()
    )

    labels = new_labels

    if changed == 0:
        print("Converged after", it + 1, "iterations")
        break

labels.orderBy("label", "id").show()

# 10. Modularity scoring: immutable original graph

Girvan-Newman repeatedly removes edges from a **working graph**.

But modularity asks how good the candidate partition is relative to the **original graph**.

Therefore keep:

```text
G0 = immutable original graph
Gt = mutable working graph used for removals/components
```

Spark's natural role in modularity scoring is:

```text
original edges
 + community label per node
 + degree per node
      ↓ joins
internal-edge counts and degree sums
      ↓ aggregation
Q
```

The important exam rule:

> Never accidentally score modularity on the already-damaged working graph.

# 11. SimRank: know where Spark stops being magic

Full SimRank stores pairwise similarities.

For \(n\) nodes:

\[
O(n^2)
\]

pair state can be required.

If \(n=100{,}000\), that is roughly \(10^{10}\) ordered pairs.

Spark does not make that state disappear.

A defensible exam pattern can be:

```text
full graph in Spark
    ↓
distributed filtering/ranking/core selection
    ↓
small explicitly bounded induced core
    ↓
pairwise SimRank on that core
    ↓
report scope honestly
```

If you reduce to 500 nodes, your result describes the **500-node core**, not automatically the entire graph.

## 11.1 SimRank preprocessing example

A common first step is to build an in-neighbor representation or predecessor pairs.

The algorithm notebook contains the full recurrence.

The Spark lesson is simply:

> Pairwise state is the bottleneck; reduce deliberately rather than casually collecting the whole graph.

# 12. Spectral partitioning: sparse matvec, not dense matrix

The graph lesson uses sparse edge-based operations.

Do **not** build a dense \(n\times n\) adjacency/Laplacian matrix for a massive sparse graph if the question emphasizes scalability.

Conceptual normalized-adjacency multiplication:

```text
current node vector x
      JOIN edges
      ↓
send normalized contribution along edges
      ↓
GROUP BY destination
      ↓
sum = Sx
```

Then spectral iteration can repeatedly use sparse matrix-vector products.

The same Spark primitives appear again:

\[
\boxed{\text{JOIN} \rightarrow \text{GROUP BY} \rightarrow \text{SUM}}
\]

# 13. A small RDD bridge — only because exam solutions may use it

DataFrames are the primary focus of this tutorial.

However, PySpark RDDs are still built-in Spark and may appear in older/passing notebooks.

You only need a minimal reading vocabulary:

```python
rdd.map(...)
rdd.filter(...)
rdd.flatMap(...)
rdd.reduceByKey(...)
rdd.join(...)
rdd.collect()
```

The same scalability rule applies:

> An RDD is distributed until you bring a large result to the driver.

If you can solve the problem cleanly with DataFrames, that is often easier to inspect and defend.

In [ ]:
pair_rdd = spark.sparkContext.parallelize([
    ("A", 1),
    ("A", 2),
    ("B", 3),
])

pair_rdd.reduceByKey(lambda a, b: a + b).collect()

The final `collect()` above is safe because the toy result is tiny.

The same call on a huge full graph result could be unsafe.

# 14. Convergence: the recurring iterative pattern

Several syllabus algorithms are iterative:

- PageRank;
- HITS;
- SimRank;
- spectral methods;
- connected-component label propagation.

Generic pattern:

```text
state_t
   ↓
compute state_{t+1}
   ↓
compute SMALL residual/change scalar
   ↓
residual < tolerance?
   ├─ yes → stop
   └─ no  → continue
```

A fixed iteration count is not automatically wrong, but unless justified it is a bounded approximation.

A stronger exam answer reports:

- `MAX_IT`;
- `TOL`;
- actual stopping iteration;
- final residual.

# 15. Correctness invariants — fastest exam debugging

| Algorithm | Fast check |
|---|---|
| stable hash sample | same entity → same decision |
| reservoir | every item final inclusion probability \(k/n\) |
| Bloom | inserted item should never query false |
| HLL | duplicates should not materially change distinct estimate |
| CMS | insertion-only estimate should not be below exact count |
| PageRank | rank sum \(\approx 1\) |
| HITS | L2 norms \(\approx 1\) after normalization |
| triangle/clustering | one triangle toy → 6 ordered closed paths |
| clustering | value in \([0,1]\) |
| modularity | score on immutable original graph |
| SimRank | diagonal \(1\), symmetry |
| iterative methods | residual/convergence reported |

When code fails under time pressure, test an invariant before staring at every line.

# 16. The seven-part SDS answer scaffold

For most substantial subproblems:

1. **Algorithm match**  
   “This is a ___ problem because the prompt requires ___.”

2. **Core rule**  
   Write the formula/recurrence/invariant.

3. **Parameters**  
   State and derive them.

4. **Verification**  
   Substitute actual values and check the requirement.

5. **Scalability**  
   State where the expensive data/state lives.

6. **Implementation**  
   Use Spark-native transformations for the large part.

7. **Sanity check + interpretation**  
   Verify one invariant and explain what the result means.

For critique questions, first say what is **correct**, then identify only real errors/limitations.

# 17. Exam mini-drills — identify the Spark pattern

### Drill 1
“Keep the same 1% of devices every day.”

**Think:** `xxhash64(device_id)` + `pmod` + threshold.

### Drill 2
“Rank all pages using PageRank.”

**Think:** source-rank join → contribution → group by destination → left join nodes → residual.

### Drill 3
“Compute hub and authority.”

**Think:** destination aggregate for authority; source aggregate for hub; normalize; preserve nodes.

### Drill 4
“Count triangles.”

**Think:** clean/symmetrize → self-join 2-hop → closing-edge join.

### Drill 5
“Use SimRank on a huge graph.”

**Think:** \(O(n^2)\) pair state. Do not pretend Spark removes the quadratic state; justify a bounded core/approximation.

### Drill 6
“Run Girvan-Newman and choose the best partition.”

**Think:** working graph for edge removals, **original graph** for modularity scoring.

### Drill 7
“Run an iterative method for exactly 10 rounds.”

**Think:** if 10 is not explicitly required, add/report a convergence residual.

# 18. Two-minute pre-submission Spark audit

Before submitting a Spark subproblem, ask:

1. Did I answer the requested output?
2. Did I state the graph/data interpretation?
3. Did I derive/verify required parameters?
4. Did I accidentally collect a large table/graph?
5. Did an inner join silently drop entities/nodes?
6. Did I preserve zero-score nodes when required?
7. Did I use only allowed built-in Spark functionality?
8. Did I report convergence or clearly label fixed iterations?
9. Did I run a sanity/invariant check?
10. Did I display the requested top-k/community/metric?
11. If I used a reduced core, did I state that the result applies only to that core?
12. For Girvan-Newman/modularity, did I keep the original graph immutable?

# 19. Final Spark mental model for the exam

When you see unfamiliar Spark code, stop reading it as syntax.

Ask:

```text
WHAT ROWS EXIST RIGHT NOW?
        ↓
WHAT KEY CONNECTS THE NEXT TABLE?
        ↓
JOIN?
        ↓
WHAT VALUE DOES EACH ROW CARRY?
        ↓
GROUP BY WHICH KEY?
        ↓
WHAT AGGREGATE?
        ↓
WHICH ENTITIES MUST BE PRESERVED?
        ↓
WHAT SMALL RESULT MAY RETURN TO DRIVER?
```

For much of the SDS exam, that reasoning is more useful than memorizing dozens of Spark methods.

# 20. What to study next

After completing Parts 1–3:

1. reopen the short `SDS_00_PySpark_Core_Reference.ipynb`;
2. it should now feel like a **quick syntax sheet**, not a mysterious notebook;
3. then open the algorithm notebooks one at a time:
   - hashing/LSH;
   - stream algorithms;
   - PageRank/HITS;
   - graph algorithms;
4. for every Spark block, narrate aloud:
   - what rows enter;
   - what the join matches;
   - what gets grouped;
   - what gets aggregated;
   - what stays distributed;
   - what small value is collected.

That is the level of PySpark understanding needed for this exam.